<div style="
background: linear-gradient(135deg, #f8f9fa 0%, #edf6f9 45%, #e8eaf6 100%);
padding: 40px;
border-radius: 20px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 8px 24px rgba(0,0,0,0.08);
border: 1px solid #dce3ea;
">

  <h1 style="
  color: #5c6b8a;
  font-size: 2.2em;
  margin: 0 0 8px 0;
  letter-spacing: 1px;
  font-weight: 700;">
  🤖 CP020003 — Artificial Intelligence 2026
  </h1>

  <h2 style="
  color: #7b8fa1;
  font-size: 1.3em;
  margin: 0 0 16px 0;
  font-weight: 400;">
  Khon Kaen University
  </h2>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    👨‍🏫 <strong style="color:#6c7aa1;">Author:</strong>
    Teerapong Panboonyuen (P'Kao)
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📧 <strong style="color:#6c7aa1;">Contact:</strong>
    teerapong.pa@chula.ac.th
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    🏫 <strong style="color:#6c7aa1;">Course:</strong>
    AI 2026 @ KKU
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📦 <strong style="color:#6c7aa1;">GitHub:</strong>
    <a href="https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1"
       style="color:#5b8def; text-decoration:none;">
       CP020003_ArtificialIntelligence_2026s1
    </a>
  </p>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="
color: #6c757d;
font-size: 0.95em;
margin: 4px 0;">
📚 Built with inspiration from the open-source AI community:
<strong style="color:#7286a0;">
Python · Pandas · NumPy · scikit-learn · PyTorch · Hugging Face · Kaggle
</strong>
</p>

  <p style="
  color: #8a97a6;
  font-size: 0.9em;
  margin-top: 12px;
  font-style: italic;">
  "This notebook is open to everyone — including those who cannot afford university.
  Knowledge is for all. 🌏"
  </p>

</div>

## 💰 Week 10 — Regression in Banking: Pricing a Loan From Linear Regression to Modern AI
### CP020003 Artificial Intelligence 2026 — In-Class Notebook

Today's job is the one every consumer-lending desk does thousands of times a day: **given a borrower's application, what interest rate should we charge?** We'll build that model six different ways — from 200-year-old linear regression to a 2026-era pretrained tabular foundation model from Hugging Face — and finish with a GenAI layer that explains *why* the model priced a loan the way it did, in plain English a loan officer (or your mom 👋) can actually read.

> 🕐 **Kept short on purpose.** This whole notebook runs end-to-end in well under 30 minutes on a free Colab GPU (`Runtime → Change runtime type → T4 GPU`). We go for **concepts you can see**, not exhaustive tuning.

**The data:** ~2,200 real peer-to-peer loan applications (LendingClub-style), each with the borrower's requested amount, FICO score, income, debt ratio, employment history, etc. — and the interest rate the platform actually assigned. Our job: learn that pricing function.

- Train: `kaggle_kku_loan_data_train.csv` (has the target, `Interest.Rate`)
- Test: `kaggle_kku_loan_data_test.csv` (target withheld — like a real Kaggle leaderboard)

**The golden rule of this entire notebook:** a regression model is only as trustworthy as the *honesty* of its inputs. Every time we add a feature, we'll ask: "would this be known **before** we quote the borrower a rate?" If not, it's leakage — and a leaky model looks great in the notebook and fails on the first real applicant.

**Roadmap:**

| Level | Family | Models |
|---|---|---|
| 1 | Foundations | Cleaning messy strings (`"18.49%"`, `"720-724"`, `"10+ years"`), leakage check, EDA |
| 2 | Linear models | Linear Regression, Ridge, Lasso, ElasticNet |
| 3 | Classic ML (trees) | Decision Tree, Random Forest, Gradient Boosting, XGBoost, LightGBM, CatBoost |
| 4 | Deep learning | PyTorch MLP regressor (GPU) |
| 5 | Modern AI (2026) | TabPFN — a pretrained tabular *foundation model* from Hugging Face — plus a GenAI layer for plain-English explanations |
| 6 | Explainability | Permutation importance, SHAP, partial dependence |
| 7 | Evaluation | MAE, MSE, RMSE, MAPE, R², Explained Variance, Median AE — all at once |
| 🏆 | Scoreboard | Every model, one leaderboard |


## 0. Setup 🔧

Colab already ships `numpy`, `pandas`, `matplotlib`, `scikit-learn`, and `torch`. We add the gradient-boosting libraries, `shap` for explainability, `transformers` for the GenAI explanation layer, and `tabpfn` for the modern tabular foundation model.

In [ ]:
# Run once per Colab session — about 40-60 seconds
!pip -q install xgboost lightgbm catboost shap tabpfn transformers --upgrade

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

SEED = # Write your lucky number here
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
pd.set_option("display.max_columns", 20)

## 1. The Business Problem: Pricing Risk with Regression 🏦

A lending platform has to answer one question for every applicant: **what interest rate compensates us for the risk of lending this person money?** Charge too high and good borrowers walk away to a competitor; charge too low and the platform loses money on defaults. This is a textbook **regression** problem — the target, `Interest.Rate`, is a continuous number, not a category.

We'll treat this the way a bank's model-risk team would: start dead simple (a straight line through FICO score), and only add complexity when it earns its keep on held-out data.

## 2. Load the Data 📥

In [ ]:
YOUR_DATA_SET_NAME = # Write your dataset name here

In [ ]:
TRAIN_URL = f"https://raw.githubusercontent.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1/main/dataset/{YOUR_DATA_SET_NAME}_train.csv"
TEST_URL  = f"https://raw.githubusercontent.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1/main/dataset/{YOUR_DATA_SET_NAME}_test.csv"

train_raw = pd.read_csv(TRAIN_URL).dropna(subset=["ID"]).reset_index(drop=True)
test_raw  = pd.read_csv(TEST_URL).dropna(subset=["ID"]).reset_index(drop=True)

print("Train:", train_raw.shape, " Test:", test_raw.shape)
train_raw.head()

## 3. Cleaning: Percentages, Ranges, and Text Mess 🧹

Real bank data is never analysis-ready. Here, several columns are strings that *look* numeric:

- `Interest.Rate`, `Debt.To.Income.Ratio` → `"18.49%"` → needs the `%` stripped
- `FICO.Range` → `"720-724"` → we'll use the **midpoint**, `722`
- `Loan.Length` → `"60 months"` → needs the word stripped
- `Employment.Length` → `"10+ years"`, `"< 1 year"`, `"n/a"` → needs custom mapping
- A few numeric-looking columns contain a literal `"."` for missing values instead of a blank cell

We write small, testable parsing functions for each — this is 90% of a real regression project's actual work.

In [ ]:
def to_num(x):
    # Turn a numeric-looking string (or literal '.') into a float, else NaN
    # Write your code here
    return

def parse_percent(x):
    if pd.isna(x) or x == ".":
        return np.nan
    return float(str(x).replace("%", ""))

def parse_fico_mid(x):
    # '720-724' -> 722.0
    # Write your code here
    return

def parse_loan_length(x):
    if pd.isna(x) or x == ".":
        return np.nan
    return float(str(x).replace("months", "").strip())

def parse_employment_length(x):
    if pd.isna(x) or x == "n/a" or x == ".":
        return np.nan
    x = str(x)
    if "<" in x:
        return 0.5
    if "10+" in x:
        return 10.0
    return float(x.split()[0])

def clean(df, has_target):
    out = pd.DataFrame()
    out["ID"] = df["ID"]
    out["Amount.Requested"] = df["Amount.Requested"].apply(to_num)
    out["Amount.Funded.By.Investors"] = df["Amount.Funded.By.Investors"].apply(to_num)
    if has_target:
        out["Interest.Rate"] = df["Interest.Rate"].apply(parse_percent)
    out["Loan.Length"] = df["Loan.Length"].apply(parse_loan_length)
    out["Loan.Purpose"] = df["Loan.Purpose"]
    out["Debt.To.Income.Ratio"] = df["Debt.To.Income.Ratio"].apply(parse_percent)
    out["State"] = df["State"]
    out["Home.Ownership"] = df["Home.Ownership"]
    out["Monthly.Income"] = df["Monthly.Income"]
    out["FICO.Mid"] = df["FICO.Range"].apply(parse_fico_mid)
    out["Open.CREDIT.Lines"] = df["Open.CREDIT.Lines"].apply(to_num)
    out["Revolving.CREDIT.Balance"] = df["Revolving.CREDIT.Balance"].apply(to_num)
    out["Inquiries.in.the.Last.6.Months"] = df["Inquiries.in.the.Last.6.Months"]
    out["Employment.Length"] = df["Employment.Length"].apply(parse_employment_length)
    return out

train = clean(train_raw, has_target=True)
test  = clean(test_raw, has_target=False)

print(train.shape, test.shape)
train.isna().sum().to_frame("missing values")

## 4. ⚠️ The Leakage Check — the Most Important Cell in This Notebook

`Amount.Funded.By.Investors` looks like an innocent numeric feature. But think about the *timeline*: on a P2P platform, investors decide how much to fund **after** seeing the listing — including its interest rate. Using it to *predict* the interest rate risks feeding the model information that, operationally, doesn't exist yet at pricing time.

Let's check how suspicious it actually is before deciding whether to keep it.

In [ ]:
corr_check = train[["Amount.Funded.By.Investors", "Amount.Requested", "Interest.Rate"]].corr()
print(corr_check.round(3))

print("\nAmount.Funded.By.Investors is {:.1%} identical in value to Amount.Requested for this sample".format(
    (train["Amount.Funded.By.Investors"] == train["Amount.Requested"]).mean()
))

**Verdict:** `Amount.Funded.By.Investors` is nearly a duplicate of `Amount.Requested` (correlation ≈ 0.97) and is set *after* the rate is quoted — so it's both **redundant** and **temporally suspicious**. We'll build our main models on a *fair* feature set that excludes it, and later run a quick side-by-side to show how much (or little) it would have inflated our score. This is the habit we want you to leave class with: **question every feature's timeline, not just its correlation.**

In [ ]:
TARGET = # Write your code here

FEATURES_FAIR = [c for c in train.columns if c not in ["ID", TARGET, "Amount.Funded.By.Investors"]]
NUM_COLS = ["Amount.Requested", "Loan.Length", "Debt.To.Income.Ratio", "Monthly.Income",
            "FICO.Mid", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance",
            "Inquiries.in.the.Last.6.Months", "Employment.Length"]
CAT_COLS = ["Loan.Purpose", "State", "Home.Ownership"]

print("Numeric features:", NUM_COLS)
print("Categorical features:", CAT_COLS)

## 5. Exploratory Data Analysis 📊

Before modeling, look at the target and its most obvious driver — credit score.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(train["Interest.Rate"], bins=30, color="#2563eb")
axes[0].set_title("Distribution of Interest.Rate (%)")
axes[0].set_xlabel("Interest Rate (%)")

axes[1].scatter(train["FICO.Mid"], train["Interest.Rate"], alpha=0.3, s=10, color="#16a34a")
axes[1].set_title("FICO Score vs Interest Rate")
axes[1].set_xlabel("FICO midpoint")
axes[1].set_ylabel("Interest Rate (%)")

train.boxplot(column="Interest.Rate", by="Loan.Length", ax=axes[2])
axes[2].set_title("Interest Rate by Loan Length")
axes[2].set_xlabel("Loan length (months)")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
corr = # Write your code here
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns))); ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.columns))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im)
plt.title("Correlation with Interest.Rate")
plt.tight_layout()
plt.show()

**Reading the plots:** interest rate is roughly bell-shaped between ~6% and ~25%; FICO score has the clearest negative relationship with rate (better credit → lower rate — exactly what we'd expect from how these platforms actually price risk); and longer (60-month) loans carry higher rates than shorter (36-month) ones.

## 6. Preprocessing Pipeline & Train/Validation Split

We build one `ColumnTransformer` (impute + scale numeric, impute + one-hot encode categorical) and reuse it inside every model's `Pipeline`. This guarantees every model sees identically-prepared data, so the leaderboard comparison is fair.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

X = # Write your code here
y = # Write your code here

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED)
print("Train:", X_train.shape, " Validation:", X_val.shape)

preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler())]), NUM_COLS),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CAT_COLS),
])

## 7. Evaluation Toolkit — Many Error Metrics at Once 📏

No single metric tells the whole story. We'll track **six** every time:

| Metric | What it tells you |
|---|---|
| **MAE** | Average error in percentage points — easy to explain to a manager |
| **MSE** | Like MAE but punishes big misses harder |
| **RMSE** | Same units as the target (%), still punishes big misses |
| **MAPE** | Error as a % of the true rate — comparable across differently-priced loans |
| **R²** | Fraction of variance in the rate our model explains (1.0 = perfect) |
| **Median AE** | Typical error, robust to a few very bad predictions |

Every model below goes through the same `evaluate()` function so nothing is compared apples-to-oranges.

In [ ]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              explained_variance_score, median_absolute_error)

leaderboard = {}   # name -> dict of metrics (+ "Family" for the final chart)

def evaluate(y_true, y_pred, name, family):
    mae   = mean_absolute_error(y_true, y_pred)
    mse   = mean_squared_error(y_true, y_pred)
    rmse  = mse ** 0.5
    mape  = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
    r2    = r2_score(y_true, y_pred)
    evs   = explained_variance_score(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    leaderboard[name] = dict(MAE=mae, MSE=mse, RMSE=rmse, MAPE=mape, R2=r2, EVS=evs, MedAE=medae, Family=family)
    print(f"{name:22s} MAE={mae:.3f}  RMSE={rmse:.3f}  MAPE={mape:5.2f}%  R2={r2:.3f}  MedAE={medae:.3f}")
    return leaderboard[name]

## 8. Level 1 — Baseline: Just Guess the Average

Every model must beat this or it isn't earning its complexity.

In [ ]:
baseline_pred = np.full_like(y_val, fill_value=y_train.mean(), dtype=float)
_ = evaluate(y_val, baseline_pred, "Baseline (mean)", "Baseline")

## 9. Level 2 — Linear Regression Family 📐

The foundation of all regression: fit a straight line (or hyperplane) that minimizes squared error. **Ridge**, **Lasso**, and **ElasticNet** add a penalty on the coefficients to fight overfitting — Ridge shrinks them smoothly, Lasso can zero them out entirely (built-in feature selection), ElasticNet blends both.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

linear_models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0, random_state=SEED),
    "Lasso": Lasso(alpha=0.01, random_state=SEED),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=SEED),
}

fitted_linear = {}
for name, model in linear_models.items():
    pipe = Pipeline([("pre", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    evaluate(y_val, pipe.predict(X_val), name, "Linear")
    fitted_linear[name] = pipe

In [ ]:
# Interpretability for free: plain Linear Regression's coefficients
lin_pipe = fitted_linear["Linear Regression"]
feat_names = lin_pipe.named_steps["pre"].get_feature_names_out()
coefs = pd.Series(lin_pipe.named_steps["model"].coef_, index=feat_names).sort_values(key=abs, ascending=False)

print("Top drivers of Interest.Rate, according to plain Linear Regression:")
print(coefs.head(8).round(3))

## 10. Level 3 — Tree-Based Machine Learning 🌳

A single decision tree splits the data on the feature that reduces error the most, recursively. **Random Forest** averages many trees trained on random subsets (bagging). **Gradient Boosting / XGBoost / LightGBM / CatBoost** instead build trees *sequentially*, each one correcting the previous ones' mistakes — this family usually wins on structured, tabular data like ours.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

tree_models = {
    "Decision Tree": DecisionTreeRegressor(max_depth=5, random_state=SEED),
    "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=8, random_state=SEED, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=SEED),
    "XGBoost": XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                             random_state=SEED, verbosity=0, tree_method="hist",
                             device="cuda" if device.type == "cuda" else "cpu"),
    "LightGBM": LGBMRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                               random_state=SEED, verbosity=-1),
    "CatBoost": CatBoostRegressor(iterations=300, depth=4, learning_rate=0.05,
                                   verbose=0, random_state=SEED),
}

fitted_trees = {}
for name, model in tree_models.items():
    pipe = Pipeline([("pre", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    evaluate(y_val, pipe.predict(X_val), name, "Tree Ensemble")
    fitted_trees[name] = pipe

## 11. Level 4 — Deep Learning: A PyTorch MLP Regressor 🧠

A small feedforward neural network can learn non-linear interactions between features without us hand-engineering them. We reuse the exact same `preprocessor` so the comparison stays fair, then hand the resulting dense matrix to PyTorch and train on GPU if available.

In [ ]:
import torch.nn as nn

# Reuse the same preprocessing, fit only on the training split
Xt_train = preprocessor.fit_transform(X_train, y_train)
Xt_val = preprocessor.transform(X_val)
if hasattr(Xt_train, "toarray"):
    Xt_train, Xt_val = Xt_train.toarray(), Xt_val.toarray()

X_train_t = torch.tensor(Xt_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1).to(device)
X_val_t   = torch.tensor(Xt_val, dtype=torch.float32).to(device)

class MLPRegressor(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x)

mlp = MLPRegressor(Xt_train.shape[1]).to(device)
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()

EPOCHS = 200
history = []
for epoch in range(EPOCHS):
    mlp.train()
    optimizer.zero_grad()
    pred = mlp(X_train_t)
    loss = loss_fn(pred, y_train_t)
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    if (epoch + 1) % 40 == 0:
        print(f"epoch {epoch+1:3d}  train MSE = {loss.item():.3f}")

mlp.eval()
with torch.no_grad():
    mlp_pred = mlp(X_val_t).cpu().numpy().ravel()

evaluate(y_val, mlp_pred, "PyTorch MLP", "Deep Learning")

plt.figure(figsize=(6, 3))
plt.plot(history)
plt.title("MLP training loss (MSE)")
plt.xlabel("epoch"); plt.ylabel("MSE")
plt.show()

## 12. Level 5 — Modern AI (2026): A Pretrained Tabular Foundation Model 🤗

Every model so far had to be *trained from scratch* on our 1,760 training rows. In 2026, a new family of models changes that: **tabular foundation models**, pretrained once on millions of synthetic datasets and shared through Hugging Face, that make predictions on a *brand-new* table with a single forward pass — no gradient descent on your data required.

We use **TabPFN**, the best-known model in this family. Because it's a foundation model (like a small GPT for tables), it can be slow on very large datasets, so we run it on a manageable sample and let it use the GPU if one is available. If the download is blocked in your environment, the cell degrades gracefully and the rest of the notebook still runs.

In [ ]:
try:
    from tabpfn import TabPFNRegressor

    # TabPFN wants purely numeric input, so we reuse our fitted preprocessor's matrices
    tabpfn_device = "cuda" if device.type == "cuda" else "cpu"
    tabpfn = TabPFNRegressor(device=tabpfn_device)

    # Foundation models like this shine on small-to-medium tables — right in our wheelhouse
    tabpfn.fit(Xt_train, y_train.values)
    tabpfn_pred = tabpfn.predict(Xt_val)

    evaluate(y_val, tabpfn_pred, "TabPFN (foundation model)", "Modern AI")
except Exception as e:
    print("TabPFN unavailable in this environment, skipping this cell gracefully.")
    print("Reason:", repr(e))

### Why this matters for your students

No architecture search, no hyperparameter grid, no training loop — TabPFN was pretrained once by its authors on millions of synthetic tables and just *reads* your table at inference time (an idea called **in-context learning**, the same mechanism behind ChatGPT-style prompting, applied to spreadsheets instead of text). It won't always beat a well-tuned CatBoost on a specific dataset, but for small tabular problems where you can't afford to tune five models, it is a genuinely 2025-2026-relevant tool worth knowing exists.

## 13. GenAI Layer: Explaining the Model in Plain English 💬

A regression model that outputs "17.3%" is not useful to a loan officer or an applicant on its own. Let's close the loop: take our best model's top feature importances and have a small, free Hugging Face language model turn them into a plain-English explanation — the same pattern used by real "explainable AI" dashboards in banking today.

In [ ]:
try:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    # Grab the top drivers from CatBoost's built-in feature importance
    cat_pipe = fitted_trees["CatBoost"]
    cat_feat_names = cat_pipe.named_steps["pre"].get_feature_names_out()
    cat_importance = pd.Series(cat_pipe.named_steps["model"].get_feature_importance(), index=cat_feat_names)
    top5 = cat_importance.sort_values(ascending=False).head(5)
    top5_text = ", ".join(f"{name.split('__')[-1]} ({score:.1f}% importance)" for name, score in
                           (top5 / top5.sum() * 100).items())

    # Loading the model/tokenizer directly (instead of `pipeline(...)`) sidesteps
    # transformers-version differences in which pipeline task names are registered.
    model_name = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    explainer_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    prompt = (
        "Explain to a bank customer, in two friendly sentences and no jargon, "
        f"why their loan interest rate is mostly driven by: {top5_text}."
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    output_ids = explainer_model.generate(**inputs, max_new_tokens=80)
    explanation = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print("Top features:", top5_text)
    print("\nGenAI explanation for the customer:\n", explanation)
except Exception as e:
    print("Hugging Face text-generation model unavailable in this environment, skipping gracefully.")
    print("Reason:", repr(e))

**The takeaway to give students:** the regression model still does the actual *pricing* — GenAI never touches the number. It only sits on top, translating the model's own feature-importance output into language a non-technical person can read. This separation (numeric model decides, language model explains) is exactly how responsible AI systems in regulated industries like banking are built.

## 14. Explainability — Which Features Actually Drive the Rate? 🔍

Feature importance from a single tree can be misleading. Two more rigorous tools:

- **Permutation importance**: shuffle one column at a time and see how much validation error gets *worse* — a model-agnostic, ground-truth measure of what the model actually relies on.
- **SHAP values**: attribute each prediction's deviation from the average to individual features, with sign and magnitude — the gold standard for "why did the model predict *this specific* rate for *this specific* applicant?"

In [ ]:
from sklearn.inspection import permutation_importance

best_name = min(leaderboard, key=lambda k: leaderboard[k]["RMSE"] if leaderboard[k]["RMSE"] == leaderboard[k]["RMSE"] else np.inf)
best_pipe = fitted_trees.get(best_name, fitted_linear.get(best_name))
print("Explaining our current best model on validation RMSE:", best_name)

perm = permutation_importance(best_pipe, X_val, y_val, n_repeats=10, random_state=SEED, scoring="neg_root_mean_squared_error")
perm_series = pd.Series(perm.importances_mean, index=X_val.columns).sort_values()

plt.figure(figsize=(7, 5))
plt.barh(perm_series.index, perm_series.values, color="#2563eb")
plt.xlabel("Increase in RMSE when this feature is shuffled")
plt.title(f"Permutation Importance — {best_name}")
plt.tight_layout()
plt.show()

In [ ]:
import shap

# SHAP TreeExplainer works directly on the fitted CatBoost model
shap_pipe = fitted_trees["CatBoost"]
Xt_val_shap = shap_pipe.named_steps["pre"].transform(X_val)
shap_feat_names = shap_pipe.named_steps["pre"].get_feature_names_out()

explainer = shap.TreeExplainer(shap_pipe.named_steps["model"])
shap_values = explainer.shap_values(Xt_val_shap)

shap.summary_plot(shap_values, Xt_val_shap, feature_names=shap_feat_names, show=False, max_display=10)
plt.title("SHAP Summary — CatBoost")
plt.tight_layout()
plt.show()

**Reading a SHAP summary plot:** each dot is one loan applicant. Red = high feature value, blue = low. Position on the x-axis shows whether that feature *pushed the predicted rate up or down* for that applicant. Expect `FICO.Mid` to dominate, with high FICO (red, on the low side of most plots for a "good" feature) pushing rates *down*.

## 15. Partial Dependence — How Does FICO Score Move the Rate, Holding Everything Else Fixed?

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

fig, ax = plt.subplots(figsize=(9, 4))
PartialDependenceDisplay.from_estimator(
    best_pipe, X_val, features=["FICO.Mid", "Debt.To.Income.Ratio"], ax=ax
)
plt.suptitle(f"Partial Dependence — {best_name}")
plt.tight_layout()
plt.show()

## 🏆 Final Leaderboard — Every Model, One Scoreboard

In [ ]:
board = pd.DataFrame(leaderboard).T
board = board.sort_values("RMSE")
board_display = board[["Family", "MAE", "RMSE", "MAPE", "R2", "MedAE"]]
print(board_display.round(3))

fig, ax = plt.subplots(figsize=(9, 6))
colors = {"Baseline": "#94a3b8", "Linear": "#2563eb", "Tree Ensemble": "#16a34a",
          "Deep Learning": "#dc2626", "Modern AI": "#9333ea"}
bar_colors = [colors.get(f, "#666666") for f in board["Family"]]

ax.barh(board.index[::-1], board["RMSE"].astype(float)[::-1], color=bar_colors[::-1])
ax.set_xlabel("Validation RMSE (percentage points, lower is better)")
ax.set_title("Loan Interest Rate Prediction — All Models")

from matplotlib.patches import Patch
present_families = [f for f in colors if f in board["Family"].values]
ax.legend(handles=[Patch(color=colors[f], label=f) for f in present_families], loc="lower right")
plt.tight_layout()
plt.show()

best = board.index[0]
naive_rmse = float(board.loc["Baseline (mean)", "RMSE"])
best_rmse = float(board.loc[best, "RMSE"])
print(f"\nBest model: {best}  (RMSE {best_rmse:.3f})")
print(f"Baseline RMSE: {naive_rmse:.3f}")
print(f"→ Beat the baseline by {(1 - best_rmse / naive_rmse) * 100:.1f}%")

### How to Actually *Read* These Results (not just admire the chart)

1. **Always compare to the baseline first.** A model that barely beats "guess the average" isn't earning its complexity — or your dataset has less signal than you'd hoped.
2. **RMSE vs MAE tells you about outliers.** If RMSE is much bigger than MAE, a handful of applicants are getting badly mispriced — worth a closer look before shipping.
3. **A tiny gap between your best linear model and your best tree model** means the relationship between features and rate is *mostly* linear — added model complexity is optional, not mandatory.
4. **Explainability isn't a nice-to-have in lending.** Regulations like the U.S. Equal Credit Opportunity Act require lenders to be able to explain an adverse pricing decision — this is exactly why we ran SHAP and permutation importance, not just a leaderboard.
5. **Re-run Section 4's leakage logic on every new feature** you're ever tempted to add — it's the single most common way real-world models fail silently.

## 16. Inference in Practice — From Raw Applicant Data to a Quoted Rate 🎯

Everything above was *training and grading* a model. Let's now actually **use** it, the way it would be used in production, in two settings:

1. **Inference where we secretly know the answer** — the validation set. We already have `y_val`, so we can put predicted vs actual side by side and see exactly how close (or far) each individual quote was. This is what you do constantly while *developing* a model.
2. **Inference on genuinely new applicants** — the `test` set. Its `Interest.Rate` column was withheld from us (like a real Kaggle leaderboard, or a real applicant who hasn't been priced yet), so there is nothing to compare against — we only get to *produce* a quote. This is what a production pricing service actually does all day.

In [ ]:
# We reuse `best_pipe` and `best_name` from the Explainability section (Section 14) —
# whichever model currently has the lowest validation RMSE.
val_pred = best_pipe.predict(X_val)

comparison = X_val.copy()
comparison["Actual Rate (%)"] = y_val.values
comparison["Predicted Rate (%)"] = val_pred.round(2)
comparison["Error (pp)"] = (comparison["Predicted Rate (%)"] - comparison["Actual Rate (%)"]).round(2)

print(f"Model: {best_name}\n")
print("10 random validation applicants — quoted rate vs what they actually received:")
comparison[["FICO.Mid", "Amount.Requested", "Loan.Purpose",
            "Actual Rate (%)", "Predicted Rate (%)", "Error (pp)"]].sample(10, random_state=SEED)

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_val, val_pred, alpha=0.4, s=15, color="#2563eb")

lims = [min(y_val.min(), val_pred.min()) - 1, max(y_val.max(), val_pred.max()) + 1]
plt.plot(lims, lims, "r--", label="Perfect prediction (predicted = actual)")
plt.xlim(lims); plt.ylim(lims)
plt.xlabel("Actual Interest Rate (%)")
plt.ylabel("Predicted Interest Rate (%)")
plt.title(f"Predicted vs Actual — {best_name}")
plt.legend()
plt.tight_layout()
plt.show()

print(f"On average, our quotes miss the true rate by {leaderboard[best_name]['MAE']:.2f} percentage points "
      f"(MAE) — i.e. a true 15.00% loan typically gets quoted somewhere around "
      f"{15 - leaderboard[best_name]['MAE']:.2f}%–{15 + leaderboard[best_name]['MAE']:.2f}%.")

**Reading this plot:** every dot is one validation applicant. The closer a dot sits to the red dashed diagonal, the better that individual quote was. A tight cloud hugging the line (like ours) means the model is well-calibrated across the whole rate range — not just accurate on average, but accurate for cheap *and* expensive loans alike. Any dots far from the line are worth pulling up individually (exactly like the table above) to ask *why* — often it's an unusual combination of features the model hasn't seen much of during training.

### Now the real thing: quoting brand-new applicants

In production you never get `y_val` — you only get the raw application. Here we run our trained pipeline on the 300 held-out `test` applicants, whose true rate nobody (including us) has access to, and produce an actual quote for each one.

In [ ]:
test_pred = best_pipe.predict(test[FEATURES_FAIR])

quotes = test[["ID", "FICO.Mid", "Amount.Requested", "Loan.Purpose"]].copy()
quotes["Predicted Interest Rate (%)"] = test_pred.round(2)

print("Sample production-style quotes for 5 brand-new applicants (no ground truth exists for these):")
quotes.head(5)

In [ ]:
OUT_PATH = "loan_rate_predictions.csv"
quotes[["ID", "Predicted Interest Rate (%)"]].to_csv(OUT_PATH, index=False)
print(f"Saved {len(quotes)} predictions to {OUT_PATH} — this file is exactly what you'd hand back "
      f"to a pricing system, or submit to a Kaggle leaderboard to find out the true score.")

**The full loop, in one sentence:** train on the past (`train`), grade yourself honestly on data the model never saw (`X_val`/`y_val` — Section 16.1), then deploy the exact same pipeline on brand-new applicants (`test` — Section 16.2) where the true answer is unknown until the loan plays out. Once those real outcomes eventually come in, they become tomorrow's validation set — this feedback loop is how banks monitor a pricing model for drift over time.

## Recap — What You Just Ran ✅

| Level | Family | Models you ran | Key idea |
|---|---|---|---|
| 1 | Foundations | — | Parsing messy strings, leakage check, EDA |
| 2 | Linear | Linear Regression, Ridge, Lasso, ElasticNet | Straight-line fit + regularization |
| 3 | Tree Ensembles | Decision Tree, Random Forest, Gradient Boosting, XGBoost, LightGBM, CatBoost | Sequential/parallel trees correcting errors |
| 4 | Deep Learning | PyTorch MLP | Learn non-linear feature interactions directly |
| 5 | Modern AI (2026) | TabPFN + Hugging Face GenAI explanation | Pretrained foundation model + LLM for plain-English explanation |
| 6 | Explainability | Permutation importance, SHAP, partial dependence | Answering *why*, not just *what* |

**Exercises to try after class:**
- Add `Amount.Funded.By.Investors` back into `FEATURES_FAIR` and re-run the leaderboard — how much does RMSE improve, and is that improvement "real" or just leakage?
- Try `TabPFNRegressor` on the *full* training set instead of a sample (if your Colab GPU allows it) — does it catch up to CatBoost?
- Swap the GenAI prompt in Section 13 to explain a *specific* applicant's SHAP values instead of the model's overall top features.
- In Section 16, swap `best_pipe` for a different model (e.g. `fitted_linear["Ridge"]`) and compare its predicted-vs-actual scatter to CatBoost's — does the spread look different?

---

<div style="
background: linear-gradient(135deg, #fafafa 0%, #eef6f9 50%, #e8eaf6 100%);
padding: 30px;
border-radius: 18px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 6px 18px rgba(0,0,0,0.06);
border: 1px solid #dce3ea;
">

  <h2 style="
  color: #5c6b8a;
  margin: 0 0 12px 0;
  font-size: 1.8em;
  font-weight: 700;">
  🎉 Well Done!
  </h2>

  <p style="
  color: #495057;
  font-size: 1.05em;
  margin: 6px 0;">
  You've completed the Week 10 Notebook for
  <strong style="color:#6c7aa1;">
  CP020003 — AI 2026 @ KKU
  </strong>
  </p>

  <!--
  <p style="
  color: #6c757d;
  font-size: 0.95em;
  margin-top: 12px;">
  Next week we dive into
  <strong style="color:#5b8def;">
  Supervised Learning
  </strong>
  — scikit-learn, train/test splits, and your first ML model 🚀
  </p>
  -->

  <hr style="
  border: 1px solid #c9d6df;
  width: 50%;
  margin: 16px auto;">

  <p style="
  color: #7d8790;
  font-size: 0.9em;
  font-style: italic;
  margin-bottom: 6px;">
  "Shared freely so that everyone, everywhere, can learn AI."
  </p>

  <p style="
  color: #8a97a6;
  font-size: 0.85em;">
  — Teerapong Panboonyuen (P'Kao) · teerapong.pa@chula.ac.th
  </p>

</div>